In [2]:

# ============================================================
# PROJECT 11: DELIVERY PERFORMANCE INTELLIGENCE AGENT
# Synthetic e-commerce data | Google Colab | Gradio + Plotly
# ============================================================

!pip -q install gradio plotly pandas numpy

import pandas as pd
import numpy as np
import gradio as gr
import plotly.express as px
from datetime import datetime, timedelta

np.random.seed(42)

# ------------------------------------------------------------
# 1. GENERATE SYNTHETIC E-COMMERCE DELIVERY DATA
# ------------------------------------------------------------

def generate_data(n=500):
    warehouses = ["Hyderabad FC", "Mumbai FC", "Delhi FC", "Bengaluru FC"]
    routes = ["Metro", "Intercity", "Rural", "Express"]
    staffing = ["Adequate", "Short-staffed", "Peak-load"]
    inventory = ["In stock", "Partial stock", "Stock transfer"]
    carriers = ["Carrier A", "Carrier B", "Carrier C"]

    rows = []
    start_date = datetime(2026, 7, 1)

    for i in range(n):
        warehouse = np.random.choice(warehouses)
        route = np.random.choice(routes, p=[.42, .28, .18, .12])
        staff = np.random.choice(staffing, p=[.64, .21, .15])
        stock = np.random.choice(inventory, p=[.78, .14, .08])
        carrier = np.random.choice(carriers)

        promised = int(np.random.choice([1, 2, 3, 4, 5, 6]))
        distance = {
            "Metro": np.random.randint(5, 80),
            "Intercity": np.random.randint(80, 600),
            "Rural": np.random.randint(30, 300),
            "Express": np.random.randint(5, 250)
        }[route]

        handling = int(np.random.randint(2, 25))
        delay = 0

        # Simulated operational delay drivers
        if staff == "Short-staffed":
            delay += 2
        elif staff == "Peak-load":
            delay += int(np.random.rand() < .65)

        if stock == "Partial stock":
            delay += 1
        elif stock == "Stock transfer":
            delay += 2

        if route == "Rural":
            delay += int(np.random.rand() < .55)
        if route == "Intercity" and distance > 350:
            delay += int(np.random.rand() < .45)
        if carrier == "Carrier C":
            delay += int(np.random.rand() < .40)
        if handling >= 18:
            delay += int(np.random.rand() < .50)

        delay += int(np.random.choice([0, 0, 0, 1, 2]))
        actual = max(1, promised + delay - int(np.random.rand() < .20))
        complaint = "Yes" if actual > promised and np.random.rand() < .35 else "No"

        order_date = start_date + timedelta(days=int(np.random.randint(0, 90)))

        rows.append({
            "Order_ID": f"ORD-{10001+i}",
            "Order_Date": order_date.strftime("%Y-%m-%d"),
            "Warehouse": warehouse,
            "Route_Type": route,
            "Carrier": carrier,
            "Staffing_Level": staff,
            "Inventory_Status": stock,
            "Distance_km": distance,
            "Handling_Hours": handling,
            "Promised_Days": promised,
            "Actual_Days": actual,
            "Customer_Complaint": complaint,
            "Order_Value": int(np.random.randint(400, 15000))
        })

    return pd.DataFrame(rows)


SAMPLE_DATA = generate_data()

# ------------------------------------------------------------
# 2. VALIDATE AND PREPARE DATA
# ------------------------------------------------------------

REQUIRED = [
    "Warehouse", "Route_Type", "Staffing_Level",
    "Inventory_Status", "Promised_Days", "Actual_Days"
]

def load_data(file_path):
    df = SAMPLE_DATA.copy() if file_path is None else pd.read_csv(file_path)
    df.columns = [str(c).strip() for c in df.columns]

    aliases = {
        "warehouse": "Warehouse",
        "route_type": "Route_Type",
        "route": "Route_Type",
        "staffing_level": "Staffing_Level",
        "staffing": "Staffing_Level",
        "inventory_status": "Inventory_Status",
        "inventory": "Inventory_Status",
        "promised_days": "Promised_Days",
        "actual_days": "Actual_Days",
        "order_id": "Order_ID",
        "order_date": "Order_Date",
        "carrier": "Carrier",
        "distance_km": "Distance_km",
        "handling_hours": "Handling_Hours",
        "customer_complaint": "Customer_Complaint",
        "complaint": "Customer_Complaint",
        "order_value": "Order_Value"
    }

    rename = {}
    for col in df.columns:
        key = col.lower().replace(" ", "_")
        if key in aliases:
            rename[col] = aliases[key]
    df = df.rename(columns=rename)

    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError("Missing required columns: " + ", ".join(missing))

    defaults = {
        "Order_ID": [f"ORDER-{i+1}" for i in range(len(df))],
        "Order_Date": None,
        "Carrier": "Unknown",
        "Distance_km": 0,
        "Handling_Hours": 0,
        "Customer_Complaint": "No",
        "Order_Value": 0
    }

    for col, default in defaults.items():
        if col not in df.columns:
            df[col] = default

    for col in [
        "Promised_Days", "Actual_Days",
        "Distance_km", "Handling_Hours", "Order_Value"
    ]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Promised_Days", "Actual_Days"])
    df = df[
        (df["Promised_Days"] > 0) &
        (df["Actual_Days"] > 0)
    ].copy()

    if df.empty:
        raise ValueError("No valid orders found in the dataset.")

    df["Delay_Days"] = (
        df["Actual_Days"] - df["Promised_Days"]
    ).clip(lower=0)

    df["Is_Delayed"] = df["Actual_Days"] > df["Promised_Days"]

    df["Complaint_Flag"] = (
        df["Customer_Complaint"].astype(str).str.lower()
        .isin(["yes", "true", "1"])
    )

    return df


# ------------------------------------------------------------
# 3. ROOT-CAUSE SIGNALS, RISK SCORING AND RECOMMENDATIONS
# ------------------------------------------------------------

def recommend_action(row):
    actions = []

    if row["Delay_Days"] > 0:
        actions.append(
            "Review tracking scans and send the customer an updated ETA"
        )

    if "short" in str(row["Staffing_Level"]).lower():
        actions.append(
            "Rebalance shifts or add temporary warehouse staff"
        )
    elif "peak" in str(row["Staffing_Level"]).lower():
        actions.append(
            "Match staffing schedules to peak order volumes"
        )

    if any(x in str(row["Inventory_Status"]).lower()
           for x in ["partial", "transfer", "out"]):
        actions.append(
            "Verify available stock and prioritize pending stock transfers"
        )

    if any(x in str(row["Route_Type"]).lower()
           for x in ["rural", "intercity"]):
        actions.append(
            "Review route planning, dispatch cut-offs and transit buffers"
        )

    if row["Handling_Hours"] >= 18:
        actions.append(
            "Investigate picking and packing queues"
        )

    if row["Carrier"] == "Carrier C" and row["Delay_Days"] > 0:
        actions.append(
            "Review carrier performance and scan history"
        )

    if row["Complaint_Flag"]:
        actions.append(
            "Escalate the complaint and confirm the revised delivery date"
        )

    if not actions:
        return "Maintain the current process and monitor delivery SLA."

    # Avoid duplicate recommendations
    return " | ".join(dict.fromkeys(actions))


def analyze_records(df):
    df = df.copy()

    # Transparent heuristic score, not a trained ML prediction
    score = np.zeros(len(df), dtype=float)

    score += np.where(
        df["Delay_Days"] >= 3, 35,
        np.where(df["Delay_Days"] >= 2, 25,
                 np.where(df["Delay_Days"] >= 1, 15, 0))
    )

    score += np.where(df["Complaint_Flag"], 15, 0)

    score += np.where(
        df["Staffing_Level"].astype(str).str.lower().str.contains("short|peak"),
        15, 0
    )

    score += np.where(
        df["Inventory_Status"].astype(str).str.lower().str.contains(
            "partial|transfer|out"
        ), 15, 0
    )

    score += np.where(
        df["Route_Type"].astype(str).str.lower().str.contains("rural|intercity"),
        8, 0
    )

    score += np.where(df["Handling_Hours"] >= 18, 7, 0)

    df["Risk_Score"] = np.clip(score, 0, 100).astype(int)

    df["Risk_Level"] = np.select(
        [df["Risk_Score"] >= 60, df["Risk_Score"] >= 30],
        ["High", "Medium"],
        default="Low"
    )

    # Choose the most actionable observed signal for each order
    conditions = [
        df["Complaint_Flag"],
        df["Handling_Hours"] >= 18,
        df["Inventory_Status"].astype(str).str.lower().str.contains(
            "partial|transfer|out"
        ),
        df["Staffing_Level"].astype(str).str.lower().str.contains("short|peak"),
        df["Route_Type"].astype(str).str.lower().str.contains("rural|intercity"),
        df["Delay_Days"] > 0
    ]

    causes = [
        "Customer complaint follow-up",
        "Warehouse handling backlog",
        "Inventory availability",
        "Staffing / peak workload",
        "Route / distance complexity",
        "Delivery execution / carrier review"
    ]

    df["Root_Cause_Signal"] = np.select(
        conditions, causes, default="No major signal"
    )

    df["Recommended_Action"] = df.apply(recommend_action, axis=1)

    return df


# ------------------------------------------------------------
# 4. INTERACTIVE DASHBOARD FUNCTION
# ------------------------------------------------------------

def dashboard(file_path, warehouse, route, risk, search_text):
    try:
        df = analyze_records(load_data(file_path))

        if warehouse != "All warehouses":
            df = df[df["Warehouse"].astype(str) == warehouse]

        if route != "All routes":
            df = df[df["Route_Type"].astype(str) == route]

        if risk != "All risk levels":
            df = df[df["Risk_Level"] == risk]

        if search_text and search_text.strip():
            q = search_text.strip().lower()
            searchable = df.astype(str).apply(
                lambda col: col.str.lower().str.contains(q, na=False)
            ).any(axis=1)
            df = df[searchable]

        if df.empty:
            return (
                "### No matching orders\nTry changing your filters.",
                px.bar(title="No matching data"),
                px.pie(title="No matching data"),
                px.bar(title="No matching data"),
                px.line(title="No matching data"),
                "No alerts for this selection.",
                pd.DataFrame()
            )

        total = len(df)
        delayed = int(df["Is_Delayed"].sum())
        on_time_rate = 100 * (1 - delayed / total)
        high_risk = int((df["Risk_Level"] == "High").sum())
        complaint_count = int(df["Complaint_Flag"].sum())

        avg_delay = (
            df.loc[df["Is_Delayed"], "Delay_Days"].mean()
            if delayed else 0
        )

        summary = f"""
<div class="kpis">
 <div class="kpi"><span>Orders reviewed</span><h2>{total:,}</h2><small>Current selection</small></div>
 <div class="kpi"><span>On-time delivery</span><h2>{on_time_rate:.1f}%</h2><small>Actual vs promised</small></div>
 <div class="kpi"><span>Delayed orders</span><h2>{delayed:,}</h2><small>Avg. delay: {avg_delay:.1f} days</small></div>
 <div class="kpi"><span>High-risk orders</span><h2>{high_risk:,}</h2><small>{complaint_count} complaint flags</small></div>
</div>
"""

        # Root-cause signal chart
        cause_df = (
            df.groupby("Root_Cause_Signal")
            .size().reset_index(name="Orders")
            .sort_values("Orders", ascending=False)
        )

        cause_fig = px.bar(
            cause_df, x="Root_Cause_Signal", y="Orders",
            color="Orders", text="Orders",
            title="Operational signals to investigate"
        )
        cause_fig.update_layout(
            xaxis_title="", yaxis_title="Orders",
            coloraxis_showscale=False,
            xaxis_tickangle=-20,
            margin=dict(l=20, r=20, t=60, b=100)
        )

        # Risk distribution
        risk_df = (
            df["Risk_Level"].value_counts()
            .reindex(["Low", "Medium", "High"], fill_value=0)
            .rename_axis("Risk Level").reset_index(name="Orders")
        )

        risk_fig = px.pie(
            risk_df, names="Risk Level", values="Orders",
            hole=.58, title="Delivery risk distribution"
        )
        risk_fig.update_layout(
            margin=dict(l=15, r=15, t=55, b=15),
            legend_title=""
        )

        # Warehouse comparison
        wh_df = df.groupby("Warehouse").agg(
            Orders=("Order_ID", "count"),
            Delay_Rate=("Is_Delayed", "mean")
        ).reset_index()

        wh_df["Delay_Rate"] *= 100

        wh_fig = px.bar(
            wh_df, x="Warehouse", y="Delay_Rate",
            text=wh_df["Delay_Rate"].map(lambda x: f"{x:.1f}%"),
            title="Delivery delay rate by warehouse"
        )
        wh_fig.update_layout(
            xaxis_title="", yaxis_title="Delayed orders (%)",
            yaxis_range=[0, 100],
            margin=dict(l=20, r=20, t=60, b=30)
        )

        # Weekly trend when dates are available
        date_df = df.copy()
        date_df["Order_Date"] = pd.to_datetime(
            date_df["Order_Date"], errors="coerce"
        )
        date_df = date_df.dropna(subset=["Order_Date"])

        if not date_df.empty:
            date_df["Week"] = (
                date_df["Order_Date"].dt.to_period("W").astype(str)
            )
            trend_df = date_df.groupby("Week").agg(
                Delay_Rate=("Is_Delayed", "mean")
            ).reset_index()
            trend_df["Delay_Rate"] *= 100

            trend_fig = px.line(
                trend_df, x="Week", y="Delay_Rate",
                markers=True, title="Weekly delivery delay trend"
            )
            trend_fig.update_layout(
                xaxis_title="", yaxis_title="Delayed orders (%)",
                margin=dict(l=20, r=20, t=60, b=40)
            )
        else:
            trend_fig = px.histogram(
                df, x="Delay_Days", title="Distribution of delay days"
            )
            trend_fig.update_layout(
                xaxis_title="Delay days", yaxis_title="Orders"
            )

        # Operational alerts
        alerts = []

        if delayed / total >= .25:
            alerts.append(
                f"🔴 **SLA alert:** {delayed / total:.1%} of orders are late. "
                "Review warehouses, routes and carriers with recurring delays."
            )

        if high_risk:
            alerts.append(
                f"🟠 **Priority queue:** {high_risk} orders have high risk scores. "
                "Prioritize tracking and proactive customer updates."
            )

        if complaint_count:
            alerts.append(
                f"📣 **Customer experience:** {complaint_count} orders have "
                "complaint flags. Assign follow-up owners and confirm revised ETAs."
            )

        stock_issue_rate = df["Inventory_Status"].astype(str).str.lower().str.contains(
            "partial|transfer|out"
        ).mean()

        if stock_issue_rate >= .12:
            alerts.append(
                "📦 **Inventory signal:** Stock or transfer issues are frequent. "
                "Check stock accuracy and dispatch-ready inventory."
            )

        staffing_issue_rate = df["Staffing_Level"].astype(str).str.lower().str.contains(
            "short|peak"
        ).mean()

        if staffing_issue_rate >= .15:
            alerts.append(
                "👥 **Staffing signal:** Staffing or peak-load flags are common. "
                "Compare order arrivals with shift coverage."
            )

        if not alerts:
            alerts.append(
                "✅ No major alert thresholds were triggered. Continue monitoring SLAs."
            )

        alert_text = "\n\n".join(alerts)

        detail_columns = [
            "Order_ID", "Warehouse", "Route_Type", "Carrier",
            "Staffing_Level", "Inventory_Status", "Promised_Days",
            "Actual_Days", "Delay_Days", "Risk_Score", "Risk_Level",
            "Root_Cause_Signal", "Recommended_Action", "Customer_Complaint"
        ]

        details = df[
            [c for c in detail_columns if c in df.columns]
        ].sort_values(
            ["Risk_Score", "Delay_Days"], ascending=False
        )

        return (
            summary, cause_fig, risk_fig, wh_fig,
            trend_fig, alert_text, details
        )

    except Exception as error:
        return (
            f"### Could not analyze the dataset\n{error}",
            px.bar(title="Check dataset"),
            px.pie(title="Check dataset"),
            px.bar(title="Check dataset"),
            px.line(title="Check dataset"),
            "Check that the CSV contains the required columns.",
            pd.DataFrame()
        )



# ============================================================
# SECTION 5 — REDESIGNED LOGISTICS CONTROL CENTRE
# Replace the previous Section 5 with this complete section.
# Sections 1–4 remain unchanged.
# ============================================================

import gradio as gr
import plotly.express as px

CONTROL_CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Space+Grotesk:wght@400;500;600;700&display=swap');

:root {
  --page: #101214;
  --panel: #191d20;
  --panel2: #202529;
  --line: #343b40;
  --text: #f4f0e8;
  --muted: #a5adb2;
  --accent: #ff7139;
  --accent2: #ffc078;
}

.gradio-container {
  max-width: 1600px !important;
  background: var(--page) !important;
  color: var(--text) !important;
  font-family: 'DM Sans', sans-serif !important;
  padding: 20px !important;
}

body, .main, .wrap, .contain {
  background: var(--page) !important;
  color: var(--text) !important;
}

h1, h2, h3, label, p, span, strong, small {
  font-family: 'DM Sans', sans-serif !important;
}

h1, h2, h3 {
  color: var(--text) !important;
  letter-spacing: -0.5px;
}

label, .prose p, .prose li {
  color: #e0e3e4 !important;
}

#topbar {
  background: #191d20;
  border: 1px solid #343b40;
  border-radius: 18px;
  padding: 25px 28px;
  margin-bottom: 18px;
  position: relative;
  overflow: hidden;
}

#topbar:after {
  content: '';
  position: absolute;
  top: 0; right: 0;
  height: 4px; width: 32%;
  background: #ff7139;
}

.eyebrow {
  color: #ff986d;
  text-transform: uppercase;
  font-size: 11px;
  letter-spacing: 2.5px;
  font-weight: 700;
}

.hero-title {
  font-family: 'Space Grotesk', sans-serif !important;
  font-size: clamp(27px, 3vw, 40px);
  line-height: 1.15;
  font-weight: 700;
  margin: 12px 0;
  color: #fff8ee;
}

.hero-subtitle {
  color: #b5bdc1;
  max-width: 800px;
  line-height: 1.7;
  font-size: 14px;
}

#control-panel {
  background: #191d20;
  border: 1px solid #343b40;
  border-radius: 16px;
  padding: 17px;
}

#control-panel h3 {
  font-family: 'Space Grotesk', sans-serif !important;
}

.kpi-grid {
  display: grid;
  grid-template-columns: repeat(4, minmax(0, 1fr));
  gap: 12px;
  margin: 0 0 15px;
}

.metric {
  background: #191d20;
  border: 1px solid #343b40;
  border-radius: 14px;
  padding: 17px;
  min-width: 0;
}

.metric-label {
  color: #a5adb2;
  font-size: 11px;
  text-transform: uppercase;
  letter-spacing: 1.2px;
  font-weight: 700;
}

.metric-value {
  color: #fff8ee;
  font-family: 'Space Grotesk', sans-serif !important;
  font-size: clamp(23px, 2.3vw, 32px);
  font-weight: 700;
  margin: 10px 0 6px;
  overflow-wrap: anywhere;
}

.metric-foot {
  color: #a5adb2;
  font-size: 11px;
}

.metric-accent {
  color: #ff986d;
}

.section-kicker {
  color: #ff986d;
  font-size: 11px;
  font-weight: 700;
  letter-spacing: 1.8px;
  text-transform: uppercase;
}

.gradio-container .block,
.gradio-container .form,
.gradio-container .panel,
.gradio-container .tabitem {
  background: #191d20 !important;
  color: #f4f0e8 !important;
  border-color: #343b40 !important;
}

.gradio-container input,
.gradio-container textarea,
.gradio-container select,
.gradio-container [role="combobox"] {
  background: #101214 !important;
  color: #fff8ee !important;
  border: 1px solid #41494f !important;
  border-radius: 9px !important;
}

.gradio-container input::placeholder,
.gradio-container textarea::placeholder {
  color: #8f999f !important;
}

.gradio-container button.primary {
  background: #ff7139 !important;
  color: #171310 !important;
  border: none !important;
  font-weight: 700 !important;
}

.gradio-container button.secondary {
  background: #292f33 !important;
  color: #f4f0e8 !important;
  border: 1px solid #41494f !important;
}

.gradio-container button {
  border-radius: 9px !important;
}

.gradio-container button:hover {
  filter: brightness(1.12);
}

.gradio-container .tab-nav {
  background: #191d20 !important;
  border-bottom: 1px solid #343b40 !important;
}

.gradio-container button[role="tab"] {
  color: #a5adb2 !important;
  background: transparent !important;
}

.gradio-container button[role="tab"][aria-selected="true"] {
  color: #ff986d !important;
  border-bottom: 2px solid #ff7139 !important;
}

.gradio-container table {
  background: #191d20 !important;
  color: #f4f0e8 !important;
}

.gradio-container th {
  background: #292f33 !important;
  color: #ffb08b !important;
}

.gradio-container td {
  color: #e7e9e9 !important;
}

#footnote {
  color: #8e989e;
  font-size: 11px;
  line-height: 1.8;
  border-top: 1px solid #343b40;
  padding-top: 14px;
  margin-top: 18px;
}

footer {
  display: none !important;
}

@media(max-width: 850px) {
  .kpi-grid { grid-template-columns: repeat(2, minmax(0, 1fr)); }
  .gradio-container { padding: 9px !important; }
  #topbar { padding: 20px; }
}
"""

# Give charts a matching dark appearance
def style_chart(fig):
    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="#191d20",
        plot_bgcolor="#191d20",
        font=dict(family="DM Sans, sans-serif", color="#e9e5dd", size=12),
        title_font=dict(family="Space Grotesk, sans-serif", size=16, color="#fff8ee"),
        margin=dict(l=24, r=24, t=58, b=35),
        colorway=["#ff7139", "#ffc078", "#64d6bd", "#9aabff", "#ed9bc1"],
        xaxis=dict(gridcolor="#30373b", zerolinecolor="#30373b"),
        yaxis=dict(gridcolor="#30373b", zerolinecolor="#30373b"),
        legend=dict(bgcolor="rgba(0,0,0,0)")
    )
    return fig

# Wrapper around your existing analysis function
def redesigned_dashboard(file_path, warehouse, route, risk, search_text):
    result = dashboard(file_path, warehouse, route, risk, search_text)
    summary, cause_fig, risk_fig, wh_fig, trend_fig, alert_text, details = result

    # Style all charts consistently
    cause_fig = style_chart(cause_fig)
    risk_fig = style_chart(risk_fig)
    wh_fig = style_chart(wh_fig)
    trend_fig = style_chart(trend_fig)

    return (
        summary, cause_fig, risk_fig,
        wh_fig, trend_fig, alert_text, details
    )

# Build the new interface
with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue="orange",
        secondary_hue="amber",
        neutral_hue="slate"
    ),
    css=CONTROL_CSS,
    title="Delivery Operations | Control Centre"
) as demo:

    gr.HTML("""
    <div id="topbar">
      <div class="eyebrow">OPERATIONS INTELLIGENCE &nbsp; / &nbsp; PROJECT 11</div>
      <div class="hero-title">Delivery Operations<br>Control Centre<span style="color:#ff7139">.</span></div>
      <div class="hero-subtitle">
        A unified workspace to investigate delivery performance, surface
        operational bottlenecks and prioritize corrective actions.
        Demonstration powered by synthetic e-commerce data.
      </div>
    </div>
    """)

    with gr.Row(equal_height=False):
        # LEFT SIDEBAR
        with gr.Column(scale=1, min_width=260, elem_id="control-panel"):
            gr.Markdown("### CONTROL DESK")
            gr.Markdown(
                "<div class='section-kicker'>01 / Data source</div>"
            )

            file_input = gr.File(
                label="Import delivery records",
                file_types=[".csv"],
                type="filepath"
            )

            gr.Markdown(
                "<div class='section-kicker'>02 / Narrow your view</div>"
            )

            warehouse_filter = gr.Dropdown(
                ["All warehouses"] + sorted(SAMPLE_DATA["Warehouse"].unique().tolist()),
                value="All warehouses",
                label="Fulfilment centre"
            )

            route_filter = gr.Dropdown(
                ["All routes"] + sorted(SAMPLE_DATA["Route_Type"].unique().tolist()),
                value="All routes",
                label="Delivery route"
            )

            risk_filter = gr.Dropdown(
                ["All risk levels", "High", "Medium", "Low"],
                value="All risk levels",
                label="Risk priority"
            )

            search_input = gr.Textbox(
                label="Find an order or signal",
                placeholder="Search order, warehouse, cause..."
            )

            analyze_btn = gr.Button(
                "RUN PERFORMANCE REVIEW  →",
                variant="primary",
                size="lg"
            )

            reset_btn = gr.Button(
                "Reset all filters",
                variant="secondary"
            )

            gr.Markdown(
                """
                <div id="footnote">
                <b style="color:#ff986d">HOW TO USE</b><br>
                Start with the sample dataset. Filter by warehouse,
                route or risk level. Import a CSV to inspect another
                dataset.<br><br>
                <b style="color:#ff986d">RISK GUIDE</b><br>
                HIGH · 60–100<br>
                MEDIUM · 30–59<br>
                LOW · 0–29
                </div>
                """
            )

        # MAIN ANALYTICS WORKSPACE
        with gr.Column(scale=3, min_width=450):
            gr.HTML(
                "<div class='section-kicker'>LIVE WORKSPACE / PERFORMANCE SNAPSHOT</div>"
            )

            summary_output = gr.HTML()

            with gr.Tabs():
                with gr.Tab("01  ROOT CAUSE SIGNALS"):
                    gr.Markdown("### Where should operations investigate first?")
                    cause_output = gr.Plot()
                    warehouse_output = gr.Plot()

                with gr.Tab("02  RISK MONITOR"):
                    gr.Markdown("### Risk profile and delivery trends")
                    risk_output = gr.Plot()
                    trend_output = gr.Plot()

                with gr.Tab("03  ALERTS & ACTIONS"):
                    gr.Markdown("### Prioritized operational alerts")
                    alerts_output = gr.Markdown()

                with gr.Tab("04  ORDER INVESTIGATION"):
                    gr.Markdown("### Order-level evidence and recommended actions")
                    details_output = gr.Dataframe(
                        label="Prioritized orders",
                        interactive=False,
                        wrap=True
                    )

            gr.HTML("""
            <div id="footnote">
              DELIVERY OPERATIONS INTELLIGENCE &nbsp; • &nbsp;
              SYNTHETIC DEMONSTRATION DATA &nbsp; • &nbsp;
              RULE-BASED RISK INDICATORS
            </div>
            """)

    inputs = [
        file_input, warehouse_filter, route_filter,
        risk_filter, search_input
    ]

    outputs = [
        summary_output, cause_output, risk_output,
        warehouse_output, trend_output, alerts_output,
        details_output
    ]

    analyze_btn.click(
        fn=redesigned_dashboard, inputs=inputs, outputs=outputs
    )
    file_input.change(
        fn=redesigned_dashboard, inputs=inputs, outputs=outputs
    )
    warehouse_filter.change(
        fn=redesigned_dashboard, inputs=inputs, outputs=outputs
    )
    route_filter.change(
        fn=redesigned_dashboard, inputs=inputs, outputs=outputs
    )
    risk_filter.change(
        fn=redesigned_dashboard, inputs=inputs, outputs=outputs
    )
    search_input.submit(
        fn=redesigned_dashboard, inputs=inputs, outputs=outputs
    )

    def reset_new_ui():
        return (
            None, "All warehouses", "All routes",
            "All risk levels", "",
            *redesigned_dashboard(
                None, "All warehouses", "All routes",
                "All risk levels", ""
            )
        )

    reset_btn.click(
        fn=reset_new_ui,
        inputs=[],
        outputs=inputs + outputs
    )

    demo.load(
        fn=redesigned_dashboard, inputs=inputs, outputs=outputs
    )

demo.launch(share=True)


/tmp/ipykernel_5371/714823482.py:805: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://622e6673752f3b56e4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
